# TAREA 6
María Fernanda Gómez Flores    
Hassan     
Portafolios de Inversión

## PUNTO A

In [7]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [8]:
# Descarga de Datos
prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
                     start='2016-01-01', end='2026-10-15')['Close']

prices.isna().sum()

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_11997/1304681109.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
[*********************100%***********************]  6 of 6 completed


Ticker
CLX     0
COST    0
KHC     0
KO      0
PG      0
WMT     0
dtype: int64

In [9]:
# Calculo de rendimientos
rets=prices.pct_change().dropna()


In [10]:
# Calcular rendimiento promedio
mu=rets.mean()


In [11]:
# Calcular matriz de correlación
corr=rets.corr()
corr

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,1.000000,0.312793,0.338188,0.369140,0.520421,0.302428
COST,0.312793,1.000000,0.272970,0.394182,0.424521,0.568327
KHC,0.338188,0.272970,1.000000,0.467080,0.444602,0.275221
KO,0.369140,0.394182,0.467080,1.000000,0.613858,0.350033
PG,0.520421,0.424521,0.444602,0.613858,1.000000,0.429701
WMT,0.302428,0.568327,0.275221,0.350033,0.429701,1.000000


In [12]:
below_zero=(rets[rets<0]).fillna(0)
below_zero

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-01-06,-0.000472,-0.009254,0.000000,-0.005405,-0.009667,0.000000
2016-01-07,-0.010942,-0.022910,-0.022168,-0.016541,-0.008733,0.000000
2016-01-08,0.000000,-0.017504,-0.010014,-0.002643,-0.015678,-0.022912
2016-01-11,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
2026-10-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2026-10-09,0.000000,-0.001055,-0.009342,0.000000,0.000000,0.000000
2026-10-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [13]:
downside=np.array(below_zero.std())
downside

array([0.00998865, 0.0084987 , 0.01128803, 0.00725476, 0.0073185 ,
       0.00839043])

In [14]:
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix=(downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr
semivar_matrix

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000100,0.000027,0.000038,0.000027,0.000038,0.000025
COST,0.000027,0.000072,0.000026,0.000024,0.000026,0.000041
KHC,0.000038,0.000026,0.000127,0.000038,0.000037,0.000026
KO,0.000027,0.000024,0.000038,0.000053,0.000033,0.000021
PG,0.000038,0.000026,0.000037,0.000033,0.000054,0.000026
WMT,0.000025,0.000041,0.000026,0.000021,0.000026,0.000070


In [15]:
semivar_port=lambda w: w.T @ semivar_matrix @ w

In [16]:
# Definir valores iniciales

# w_inicial = [0.25, 0.25, 0.25, 0.25] # Equa wight
## hacer los pesos equitativos de forma automatica segun el numero de activos que tengo. 
n_assets = len(prices.columns)

w_inicial = np.ones(n_assets) / n_assets

In [17]:
# Definir las cotas
cotas = [(0,1)] * n_assets

In [18]:
# Definir las restricciones
restr = lambda w: sum(w) -1

In [19]:
# Definir tolerancia
np.random.seed(42)
tol = 1e-32

In [20]:
# Aplicar función de scipy.optimize
results = minimize(fun=semivar_port, x0=w_inicial, bounds=cotas, tol=tol,
         constraints={"fun": restr, "type": "eq"})

results

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 3.5412415953530834e-05
       x: [ 8.719e-02  1.509e-01  1.927e-02  3.343e-01  2.110e-01
            1.974e-01]
     nit: 39
     jac: [ 7.082e-05  7.082e-05  7.082e-05  7.082e-05  7.082e-05
            7.082e-05]
    nfev: 285
    njev: 39

In [22]:
# Obtener semivarianza mínima
semivar_min = semivar_port(w_min_semivar)
semivar_min

np.float64(3.5412415953530834e-05)

In [25]:
def opt_min_semivar(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular matriz de correlación
    corr = rets.corr()

    #Calcular downside risk
    below_zero = (rets[rets<0]).fillna(0)
    downside = np.array(below_zero.std())

    #Calcular matriz de semivarianza
    n_assets = len(prices.columns)
    semivar_matrix = (downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr

    # Funcion.
    semivar = lambda w: w.T @ semivar_matrix @ w

    #Variables iniciales (pesos)
    w_0 = np.ones(n_assets) / n_assets

    # Restricciones.
    restr = lambda w: sum(w) - 1

    #rangos de las variables de decisión
    rangos = [(0,1)] * n_assets

    # tolerancia
    tol = 1e-32

    #Optimizar
    results = minimize(fun=semivar, x0=w_0, bounds=rangos,
                       constraints={'fun': restr, 'type': 'eq'}, tol=tol)

    return results.x

In [26]:
# Verificar resultados de la función
w_min_semivar=opt_min_semivar(prices)
dict(zip(prices.columns,w_min_semivar))

{'CLX': np.float64(0.08719402912272797),
 'COST': np.float64(0.15088444174348326),
 'KHC': np.float64(0.01927268143763937),
 'KO': np.float64(0.33428265675441543),
 'PG': np.float64(0.2109791088062078),
 'WMT': np.float64(0.19738708213552608)}

In [27]:
# Obtener semivarianza mínima
semivar_port(w_min_semivar)

np.float64(3.5412415953530834e-05)

In [28]:
# Simular portafolios con montecarlo
n_sim=10000
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [29]:
# Obtener la mínima semivarianza
semivars_port=[semivar_port(w_i) for w_i in w_random]
np.min(semivars_port)

np.float64(3.563832666358864e-05)

In [30]:
# Obtener los pesos del portafolio con mínima semivarianza
w_semivar=w_random[np.argmin(semivars_port)]
dict(zip(prices.columns, w_semivar))

{'CLX': np.float64(0.04523198269375449),
 'COST': np.float64(0.18032970063786),
 'KHC': np.float64(0.02431282278193316),
 'KO': np.float64(0.34349231541965103),
 'PG': np.float64(0.19048463084264106),
 'WMT': np.float64(0.2161485476241604)}

In [31]:
# Comparar resultados
comparacion=pd.DataFrame({
    'SciPy':w_min_semivar*100,
    'Montecarlo':w_semivar*100
},index=prices.columns)

comparacion

,SciPy,Montecarlo
Ticker,,
CLX,8.719403,4.523198
COST,15.088444,18.032970
KHC,1.927268,2.431282
KO,33.428266,34.349232
PG,21.097911,19.048463
WMT,19.738708,21.614855


## PUNTO B

## PUNTO C

## EXTRA